# `tbl-results-mse`: OLS treatment-effect error and empirical cross-entropy by dataset and regime

In [23]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "tbl-results-mse"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/tbl-results-mse.*


## Run design

In [24]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
BASE_SLUG = "row1_base_untrained"
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
VALIDATORS = ("kriner2026", "grummon2025")

ctx = ma.load_microdata_context(
    run_roots=RUN_ROOTS,
    models=MODELS,
    base_slug=BASE_SLUG,
    trained_slugs=TRAINED_SLUGS,
    validators=VALIDATORS,
)
display(ctx.status_grid)

,kev-0.8b,kev-4b,kev-9b
regime,,,
row1_base_untrained,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
row2b_wu_ce_oracle,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
row3_berke_ce_only,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
row5_full_berke_wu_ce_oracle,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]


## Labels and formatting (edit wording here)

In [25]:
MODEL_LABELS = {"kev-0.8b": "KEV-0.8B", "kev-4b": "KEV-4B", "kev-9b": "KEV-9B"}

# Row order follows this dict.
REGIME_LABELS = {
    BASE_SLUG: "Off-the-shelf",
    "row2b_wu_ce_oracle": "Electricity",
    "row3_berke_ce_only": "Drones",
    "row5_full_berke_wu_ce_oracle": "Drones + Electricity",
}

# Column order follows this dict. Keys are the dataset prefixes in ctx.regimes_by_model columns.
DATASET_LABELS = {
    "kriner": "Flights",
    "kantorowicz": "Consumer rights",
    "magistro": "AI & offshoring",
    "grummon": "Food labels",
    "covid": "Vaccines",
}

# Sub-columns under each dataset. Keys are the metric suffixes in ctx.regimes_by_model columns ("kl" holds empirical CE).
METRIC_LABELS = {"mse": r"$\mathrm{MSE}_\beta$", "kl": "CE"}
METRIC_SCALE = {"mse": 1000.0, "kl": 1.0}
METRIC_DECIMALS = {"mse": 2, "kl": 3}

# Each model gets its own row spanning the full table, above its regime rows.
MODEL_ROW_TEMPLATE = "*{model_label}*"
STUB_HEADERS = ("Regime",)

## Build the cells

In [26]:
# Bold marks the lowest value per model, dataset, and metric.
# table_body lists the body rows in order: ("model", text) for a model row, ("data", cells) for a regime row.
table_body = []
rows = []
display_rows = []
for model_slug in MODELS:
    regimes_df = ctx.regimes_by_model[model_slug]
    cell_text = {}
    for dataset_key in DATASET_LABELS:
        for metric_key in METRIC_LABELS:
            column = f"{dataset_key}_{metric_key}"
            values = regimes_df.loc[list(REGIME_LABELS), column].astype(float)
            if values.isna().any():
                raise ValueError(f"{model_slug}: missing {column} for {list(values[values.isna()].index)}")
            best_slug = values.idxmin()
            for regime_slug in REGIME_LABELS:
                scaled = float(values.loc[regime_slug]) * METRIC_SCALE[metric_key]
                text = f"{scaled:.{METRIC_DECIMALS[metric_key]}f}"
                if regime_slug == best_slug:
                    text = f"**{text}**"
                cell_text[(regime_slug, dataset_key, metric_key)] = text

    model_row_text = MODEL_ROW_TEMPLATE.format(model_label=MODEL_LABELS[model_slug])
    table_body.append(("model", model_row_text))
    for regime_slug, regime_label in REGIME_LABELS.items():
        row = [regime_label]
        for dataset_key in DATASET_LABELS:
            for metric_key in METRIC_LABELS:
                row.append(cell_text[(regime_slug, dataset_key, metric_key)])
        table_body.append(("data", row))
        rows.append(row)
        display_rows.append([MODEL_LABELS[model_slug], *row])

sub_headers = list(STUB_HEADERS)
for dataset_key in DATASET_LABELS:
    for metric_key in METRIC_LABELS:
        sub_headers.append(METRIC_LABELS[metric_key])
display(pd.DataFrame(display_rows, columns=["Model", *sub_headers]))

,Model,Regime,$\mathrm{MSE}_\beta$,CE,$\mathrm{MSE}_\beta$,CE,$\mathrm{MSE}_\beta$,CE,$\mathrm{MSE}_\beta$,CE,$\mathrm{MSE}_\beta$,CE
0,KEV-0.8B,Off-the-shelf,17.75,1.086,8.34,1.418,33.92,2.283,5.46,1.059,12.07,1.185
1,KEV-0.8B,Electricity,8.91,0.693,7.35,0.821,29.60,1.689,3.53,0.915,3.73,1.082
2,KEV-0.8B,Drones,9.68,0.709,7.17,0.806,34.41,1.630,3.25,0.953,4.08,1.105
3,KEV-0.8B,Drones + Electricity,**4.62**,**0.647**,**6.04**,**0.695**,**26.61**,**1.558**,**2.30**,**0.879**,**2.56**,**1.067**
4,KEV-4B,Off-the-shelf,4.10,0.811,4.89,0.779,99.01,2.434,21.10,1.100,9.08,1.525
5,KEV-4B,Electricity,1.72,0.625,3.27,0.650,113.26,1.886,3.57,0.887,1.73,1.120
6,KEV-4B,Drones,1.00,0.623,2.64,0.663,74.69,1.797,7.04,0.959,4.63,1.195
7,KEV-4B,Drones + Electricity,**0.73**,**0.588**,**2.25**,**0.629**,**63.98**,**1.636**,**2.28**,**0.854**,**1.12**,**1.046**
8,KEV-9B,Off-the-shelf,5.12,0.833,1.76,0.804,82.89,2.437,22.73,1.238,14.30,1.422
9,KEV-9B,Electricity,2.17,0.647,**1.34**,0.694,95.68,1.873,18.85,0.972,4.25,1.203


## Format as a Pandoc grid table

In [27]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\rho": "ρ", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    surname = match.group(1).title()
    year = match.group(2)
    return f"({surname} {year})"


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    text = cell
    if is_bold:
        text = text[2:-2]
    text = re.sub(r"\[@([a-z]+)(\d{4})[a-z0-9]*\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt

In [28]:
# Pandoc grid table: dataset names span their metric sub-columns in the first header row.
n_stubs = len(STUB_HEADERS)
n_metrics = len(METRIC_LABELS)
n_columns = len(sub_headers)


def span_columns(dataset_index):
    first = n_stubs + n_metrics * dataset_index
    return list(range(first, first + n_metrics))


# Grid cells must hold their raw markdown, so convert the point widths to characters with the smallest
# factor that fits every cell and every spanned dataset name; the proportions between columns stay the same.
widths_pt = column_widths_pt(sub_headers, rows)
chars_per_pt = 0.0
for column_index in range(n_columns):
    widest_raw = len(sub_headers[column_index])
    for row in rows:
        widest_raw = max(widest_raw, len(row[column_index]))
    chars_per_pt = max(chars_per_pt, widest_raw / widths_pt[column_index])
for dataset_index, dataset_label in enumerate(DATASET_LABELS.values()):
    span_pt = 0.0
    for column_index in span_columns(dataset_index):
        span_pt += widths_pt[column_index]
    chars_per_pt = max(chars_per_pt, len(dataset_label) / span_pt)
widths = []
for width_pt in widths_pt:
    widths.append(math.ceil(width_pt * chars_per_pt))


def span_inner_width(dataset_index):
    columns = span_columns(dataset_index)
    return sum(widths[c] for c in columns) + 3 * (len(columns) - 1)


def padded(text, width):
    return " " + text.ljust(width) + " "


top_parts = [("-" * (widths[c] + 2)) for c in range(n_stubs)]
for dataset_index in range(len(DATASET_LABELS)):
    top_parts.append("-" * (span_inner_width(dataset_index) + 2))
top_border = "+" + "+".join(top_parts) + "+"

header_1_parts = [padded(STUB_HEADERS[c], widths[c]) for c in range(n_stubs)]
for dataset_index, dataset_label in enumerate(DATASET_LABELS.values()):
    header_1_parts.append(padded(dataset_label, span_inner_width(dataset_index)))
header_1 = "|" + "|".join(header_1_parts) + "|"

stub_blanks = [" " * (widths[c] + 2) for c in range(n_stubs)]
metric_rules = ["-" * (widths[c] + 2) for c in range(n_stubs, n_columns)]
header_middle = "|" + "|".join(stub_blanks) + "+" + "+".join(metric_rules) + "+"

metric_cells = [padded(sub_headers[c], widths[c]) for c in range(n_stubs, n_columns)]
header_2 = "|" + "|".join(stub_blanks + metric_cells) + "|"

header_end_parts = []
for c in range(n_columns):
    if c < n_stubs:
        header_end_parts.append(":" + "=" * (widths[c] + 1))
    else:
        header_end_parts.append("=" * (widths[c] + 1) + ":")
header_end = "+" + "+".join(header_end_parts) + "+"

row_rule = "+" + "+".join("-" * (widths[c] + 2) for c in range(n_columns)) + "+"

# A model row is one cell spanning every column: no inner | on its line, column + marks on the rules around it.
full_inner_width = sum(widths) + 3 * (n_columns - 1)

lines = [top_border, header_1, header_middle, header_2, header_end]
for row_kind, row_content in table_body:
    if row_kind == "model":
        lines.append("|" + padded(row_content, full_inner_width) + "|")
    elif row_kind == "data":
        lines.append("|" + "|".join(padded(row_content[c], widths[c]) for c in range(n_columns)) + "|")
    else:
        raise ValueError(f"Unknown body row kind {row_kind!r}")
    lines.append(row_rule)
table_md = "\n".join(lines) + "\n"

## Save `tbl-results-mse.md` (included by the manuscript)

In [30]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

+----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+
| Regime                                 | Flights                                  | Consumer rights                          | AI & offshoring                          | Food labels                              | Vaccines                                 |
|                                        +----------------------+-------------------+----------------------+-------------------+----------------------+-------------------+----------------------+-------------------+----------------------+-------------------+
|                                        | $\mathrm{MSE}_\beta$ | CE                | $\mathrm{MSE}_\beta$ | CE                | $\mathrm{MSE}_\beta$ | CE                | $\mathrm{MSE}_\beta$ | CE                | $\mathrm{MS